In [10]:
from langchain_openai import ChatOpenAI
from langchain import tools
from dotenv import load_dotenv
from langchian_core.prompts import ChatPromptTemplate
import os
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API key is set.")
else: 
    print("API key is not set.")   


ModuleNotFoundError: No module named 'langchian_core'

In [3]:
llm=ChatOpenAI(model="gpt-5-nano", temperature=0)

In [4]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def search_duckduckgo(query: str) -> str:
    """This tool seraches DuckDuckGo got given query"""
    duck_search = DuckDuckGoSearchRun()
    return duck_search.invoke(query)


In [5]:
from langchain_community.tools  import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
@tool
def search_wikipedia(query: str) -> str:
    """This tool seraches Wikipedia got given query"""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    wiki_query.invoke(query)




In [6]:
@tool
def personal_info(name:str):
    "use this tool to get personal info about a person"
    info ={
        "alice":"software dev",
        "kiya":"software tester"
    }

    return info.get(name,"No info available")


In [8]:
tools =[search_duckduckgo,search_wikipedia,personal_info]
llm_with_tools=llm.bind_tools(tools)

In [9]:
from typing import TypedDict,List

class graph_schema(TypedDict):
    messages:List

In [17]:
def llm_node(state: graph_schema) -> graph_schema:
    messages = state["messages"]

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "you are a helpful assistant"),
            ("human", "{input}")
        ]
    )

    # LLM with tools
    chain = prompt | llm_with_tools

    response = chain.invoke({"input": messages})

    state["messages"] = messages + [response]

    return state


In [19]:
from langgraph.prebuilt import ToolNode
tool_node = ToolNode(llm_node)
initial_state = {
    "messages":[]
}
final_state = tool_node.invoke(initial_state)
print(final_state["messages"])
llm_with_tools.invoke("what is personal info about alice?")





TypeError: 'function' object is not iterable